In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))



In [ ]:
from src.configs.config import *
from src.dataset.loader import TankriDataset
from src.dataset.augmentation import train_transform, val_transform
from src.models import SimpleCNN
from src.training.train import train
from src.evaluation.evaluate import evaluate
from src.utils.utils import set_seed

import json
import mlflow
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from torch import nn, optim

In [ ]:
import src.configs.config as config
EPOCHS = config.EPOCHS
BATCH_SIZE = config.BATCH_SIZE
RANDOM_SEED = config.RANDOM_SEED
IMAGE_SIZE = config.IMAGE_SIZE


In [ ]:
set_seed(RANDOM_SEED)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

In [ ]:
df = pd.read_csv(LABELS_FILE)

print(df.head())
counts = df["label"].value_counts()

# Keep only classes with at least 2 samples
valid_classes = counts[counts >= 2].index

df_filtered = df[df["label"].isin(valid_classes)].copy()
print("------------------------")
print(df_filtered.head())
print(f"Total Images : {len(df)}")
print(f"Total Classes : {df_filtered['label'].nunique()}")

In [ ]:
from src.utils.label_mapping import create_label_mapping, save_label_mapping

label_to_idx, idx_to_label = create_label_mapping(LABELS_FILE)
save_label_mapping(label_to_idx, idx_to_label, "../artifacts")

NUM_CLASSES = len(label_to_idx)
print(NUM_CLASSES)

In [ ]:
print(df["label"].unique())
target_char = "𑚖𑚷"

matches = df.loc[df["label"] == target_char, "image"] 

print(matches.tolist())
df["label"].value_counts()

In [ ]:
train_df, val_df = train_test_split(
    df_filtered,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=df_filtered["label"],
)

In [ ]:
from src.models import ResNet18Model
from src.dataset.augmentation import (
    train_transform_resnet,
    val_transform_resnet,
)
train_dataset = TankriDataset(
    dataframe=train_df,
    image_dir=IMAGES_DIR,
    label_to_idx=label_to_idx,
    transform=train_transform_resnet,
)

val_dataset = TankriDataset(
    dataframe=val_df,
    image_dir=IMAGES_DIR,
    label_to_idx=label_to_idx,
    transform=val_transform_resnet,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

In [ ]:
# Instantiate model dynamically using config fields
if config.MODEL_NAME == "ResNet18":
    from src.models import ResNet18Model
    model = ResNet18Model(
        num_classes=NUM_CLASSES,
        pretrained=config.PRETRAINED,
        dropout=config.DROPOUT,
        unfreeze_layer3=config.UNFREEZE_LAYER3,
        unfreeze_layer4=config.UNFREEZE_LAYER4,
    )
elif config.MODEL_NAME == "SimpleCNN":
    from src.models import SimpleCNN
    model = SimpleCNN(
        num_classes=NUM_CLASSES,
        dropout=config.DROPOUT,
    )
else:
    raise ValueError(f"Unknown model name: {config.MODEL_NAME}")

model = model.to(device)


In [ ]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch.nn as nn
import torch.optim as optim

criterion = nn.CrossEntropyLoss(label_smoothing=config.LABEL_SMOOTHING)
optimizer = optim.Adam(
    model.parameters(),
    lr=config.LEARNING_RATE,
)

scheduler = CosineAnnealingLR(
    optimizer,
    T_max=config.EPOCHS,
)


In [ ]:
import mlflow
from src.utils.mlflow_init import init_mlflow

init_mlflow(config.EXPERIMENT_NAME)


In [ ]:
with mlflow.start_run(run_name=config.EXPERIMENT_NAME):

    history = train(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        epochs=config.EPOCHS,
        device=device,
        scheduler=scheduler,
        save_best=True,
        save_dir="models",
        image_size=config.IMAGE_SIZE,
    )

    for epoch in range(config.EPOCHS):
        mlflow.log_metric("train_loss", history["train_loss"][epoch], step=epoch)
        mlflow.log_metric("train_accuracy", history["train_accuracy"][epoch], step=epoch)
        mlflow.log_metric("val_loss", history["val_loss"][epoch], step=epoch)
        mlflow.log_metric("val_accuracy", history["val_accuracy"][epoch], step=epoch)

    # Log evaluation artifacts via the evaluation module
    from src.evaluation.evaluate import log_evaluation_artifacts
    log_evaluation_artifacts(
        model=model,
        val_loader=val_loader,
        device=device,
        save_dir="models",
        history=history,
        label_to_idx=label_to_idx,
        idx_to_label=idx_to_label,
    )


In [ ]:
import numpy as np

model.eval()
predictions = []
true_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        predictions.extend(preds.cpu().tolist())
        true_labels.extend(labels.cpu().tolist())

predictions = np.array(predictions)
true_labels = np.array(true_labels)

print("Per-class validation accuracy:")
print("-" * 60)

for cls in range(NUM_CLASSES):
    mask = true_labels == cls
    if mask.sum() == 0:
        continue

    correct = (predictions[mask] == cls).sum()
    total = mask.sum()
    accuracy = correct / total

    print(
        f"{idx_to_label[cls]}: {accuracy:.2%} ({correct}/{total})"
    )

print("-" * 60)

worst_classes = []
for cls in range(NUM_CLASSES):
    mask = true_labels == cls
    if mask.sum() == 0:
        continue

    correct = (predictions[mask] == cls).sum()
    total = mask.sum()
    accuracy = correct / total
    worst_classes.append((accuracy, cls, correct, total))

worst_classes.sort(key=lambda x: x[0])

print("Worst-performing classes:")
for accuracy, cls, correct, total in worst_classes[:10]:
    print(
        f"{idx_to_label[cls]}: {accuracy:.2%} ({correct}/{total})"
    )


In [ ]:
import torch

torch.save(model.state_dict(), "resnet18_baseline.pth")

mlflow.log_artifact("resnet18_baseline.pth")

In [ ]:
img, label = train_dataset[0]

print(img.shape)

In [ ]:
trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total = sum(
    p.numel()
    for p in model.parameters()
)

print(f"Trainable: {trainable:,}")
print(f"Total: {total:,}")

In [ ]:
import cv2
import matplotlib.pyplot as plt

img = cv2.imread(
    str(IMAGES_DIR / "1.png"),
    cv2.IMREAD_GRAYSCALE,
)

_, thresh = cv2.threshold(
    img,
    240,
    255,
    cv2.THRESH_BINARY_INV,
)

coords = cv2.findNonZero(thresh)

print(coords is None)

if coords is not None:
    x, y, w, h = cv2.boundingRect(coords)
    print(x, y, w, h)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(img, cmap="gray")
plt.title("Original")

plt.subplot(1, 2, 2)
plt.imshow(thresh, cmap="gray")
plt.title("Threshold")

plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

model.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        predictions = outputs.argmax(dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predictions.cpu().numpy())

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predictions.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

present_labels = sorted(np.unique(np.concatenate([y_true, y_pred])))
present_target_names = [idx_to_label[idx] for idx in present_labels]

print(
    classification_report(
        y_true,
        y_pred,
        labels=present_labels,
        target_names=present_target_names,
        digits=3,
        zero_division=0,
    )
)


In [ ]:
cm = confusion_matrix(y_true, y_pred)

class_accuracy = cm.diagonal() / cm.sum(axis=1)

for idx, acc in enumerate(class_accuracy):
    print(
        f"{idx_to_label[idx]} : {acc:.2%}"
    )

In [ ]:
results = []

for idx, acc in enumerate(class_accuracy):
    results.append(
        (
            idx_to_label[idx],
            acc
        )
    )

results = sorted(
    results,
    key=lambda x: x[1]
)

print("Worst performing classes:\n")

for label, acc in results[:10]:
    print(label, f"{acc:.2%}")